# Poisson Baseline Model

Цель: подготовить простой baseline для предсказания `lambdaHome` и `lambdaAway`.

Используем Poisson Regression с набором признаков:

- `team`
- `opponent`
- `is_home`
- `league`
- `season`

Исходный датасет содержит матчи в формате:

- `home_team`
- `away_team`
- `home_goals`
- `away_goals`
- `home_xg`
- `away_xg`
- `date`
- `league`
- `season`

Для обучения Poisson-регрессии один матч будем превращать в две строки:

- строка для домашней команды
- строка для гостевой команды

### Imports

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

### Load data

In [4]:
PROJECT_ROOT = Path("..").resolve()

DATA_DIR = PROJECT_ROOT / "data"
DATA_PATH = DATA_DIR / "matches_clean.csv"

DATA_PATH

matches = pd.read_csv(DATA_PATH)

matches.head()

,understat_id,home_team,away_team,home_goals,away_goals,home_xg,away_xg,date,league,season
0,13977,Bordeaux,Nantes,0,0,0.600075,0.187821,2020-08-21,Ligue_1,2020
1,13978,Dijon,Angers,0,1,0.739709,2.022140,2020-08-22,Ligue_1,2020
2,13979,Lille,Rennes,1,1,0.400974,1.295180,2020-08-22,Ligue_1,2020
3,13982,Monaco,Reims,2,2,2.676220,1.832570,2020-08-23,Ligue_1,2020
4,13980,Lorient,Strasbourg,3,1,3.072140,0.326826,2020-08-23,Ligue_1,2020


### Info

In [5]:
print(f"Rows: {matches.shape[0]}")
print(f"Columns: {matches.shape[1]}")

matches.columns.tolist()

matches.info()
print()

key_columns = [
    "understat_id",
    "date",
    "league",
    "season",
    "home_team",
    "away_team",
    "home_goals",
    "away_goals",
    "home_xg",
    "away_xg",
]

matches[key_columns].head(10)

print("Leagues:")
display(matches["league"].value_counts())

print()

print("Seasons:")
display(matches["season"].value_counts().sort_index())

Rows: 8982
Columns: 10
<class 'pandas.DataFrame'>
RangeIndex: 8982 entries, 0 to 8981
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   understat_id  8982 non-null   int64  
 1   home_team     8982 non-null   str    
 2   away_team     8982 non-null   str    
 3   home_goals    8982 non-null   int64  
 4   away_goals    8982 non-null   int64  
 5   home_xg       8982 non-null   float64
 6   away_xg       8982 non-null   float64
 7   date          8982 non-null   str    
 8   league        8982 non-null   str    
 9   season        8982 non-null   int64  
dtypes: float64(2), int64(4), str(4)
memory usage: 701.8 KB

Leagues:


league
La_Liga       1900
EPL           1900
Serie_A       1900
Ligue_1       1752
Bundesliga    1530
Name: count, dtype: int64


Seasons:


season
2020    1826
2021    1826
2022    1826
2023    1752
2024    1752
Name: count, dtype: int64

## Feature engineering

In [6]:
df = matches.copy()

df["date"] = pd.to_datetime(df["date"])

# Строки для домашних команд
home_rows = df[
    [
        "understat_id",
        "date",
        "league",
        "season",
        "home_team",
        "away_team",
        "home_goals",
    ]
].copy()

# Для домашней команды:
# home_team -> team
# away_team -> opponent
# home_goals -> goals
# is_home = 1
home_rows = home_rows.rename(
    columns={
        "home_team": "team",
        "away_team": "opponent",
        "home_goals": "goals",
    }
)

home_rows["is_home"] = 1



# Строки для гостевых команд
away_rows = df[
    [
        "understat_id",
        "date",
        "league",
        "season",
        "away_team",
        "home_team",
        "away_goals",
    ]
].copy()

# Для гостевой команды:
# away_team -> team
# home_team -> opponent
# away_goals -> goals
# is_home = 0
away_rows = away_rows.rename(
    columns={
        "away_team": "team",
        "home_team": "opponent",
        "away_goals": "goals",
    }
)

away_rows["is_home"] = 0



# Объединяем в один датасет
team_matches = pd.concat(
    [home_rows, away_rows],
    ignore_index=True,
)

# Оставляем только нужные для baseline колонки
team_matches = team_matches[
    [
        "understat_id",
        "date",
        "league",
        "season",
        "team",
        "opponent",
        "is_home",
        "goals",
    ]
]

# Сортируем по времени
team_matches = team_matches.sort_values(
    ["date", "understat_id", "is_home"],
    ascending=[True, True, False],
).reset_index(drop=True)

print(f"Original matches: {len(df)}")
print(f"Team-match rows: {len(team_matches)}")
print(f"Expected rows: {len(df) * 2}")

team_matches.head(10)

Original matches: 8982
Team-match rows: 17964
Expected rows: 17964


,understat_id,date,league,season,team,opponent,is_home,goals
0,13977,2020-08-21,Ligue_1,2020,Bordeaux,Nantes,1,0
1,13977,2020-08-21,Ligue_1,2020,Nantes,Bordeaux,0,0
2,13978,2020-08-22,Ligue_1,2020,Dijon,Angers,1,0
3,13978,2020-08-22,Ligue_1,2020,Angers,Dijon,0,1
4,13979,2020-08-22,Ligue_1,2020,Lille,Rennes,1,1
5,13979,2020-08-22,Ligue_1,2020,Rennes,Lille,0,1
6,13980,2020-08-23,Ligue_1,2020,Lorient,Strasbourg,1,3
7,13980,2020-08-23,Ligue_1,2020,Strasbourg,Lorient,0,1
8,13982,2020-08-23,Ligue_1,2020,Monaco,Reims,1,2
9,13982,2020-08-23,Ligue_1,2020,Reims,Monaco,0,2


## Train / validation / test split

- `train`: сезоны 2020, 2021, 2022
- `validation`: сезон 2023
- `test`: сезон 2024

In [7]:
train_seasons = [2020, 2021, 2022]
valid_seasons = [2023]
test_seasons = [2024]

train_df = team_matches[team_matches["season"].isin(train_seasons)].copy()
valid_df = team_matches[team_matches["season"].isin(valid_seasons)].copy()
test_df = team_matches[team_matches["season"].isin(test_seasons)].copy()

print(f"Train rows: {len(train_df)}")
print(f"Valid rows: {len(valid_df)}")
print(f"Test rows:  {len(test_df)}")

Train rows: 10956
Valid rows: 3504
Test rows:  3504


### Проверка распределений

In [8]:
# Проверяем, что во всех split'ах есть нужные лиги

split_league_distribution = pd.DataFrame(
    {
        "train": train_df["league"].value_counts(),
        "valid": valid_df["league"].value_counts(),
        "test": test_df["league"].value_counts(),
    }
).fillna(0).astype(int)

split_league_distribution

# Проверяем, что split действительно сделался по сезонам

split_season_distribution = pd.DataFrame(
    {
        "train": train_df["season"].value_counts().sort_index(),
        "valid": valid_df["season"].value_counts().sort_index(),
        "test": test_df["season"].value_counts().sort_index(),
    }
).fillna(0).astype(int)

split_season_distribution

# Проверяем, какие команды из validation/test уже были в train

train_teams = set(train_df["team"])
valid_teams = set(valid_df["team"])
test_teams = set(test_df["team"])

valid_known_teams = valid_teams & train_teams
valid_unknown_teams = valid_teams - train_teams

test_known_teams = test_teams & train_teams
test_unknown_teams = test_teams - train_teams

print(f"Train teams: {len(train_teams)}")

print()
print(f"Validation teams: {len(valid_teams)}")
print(f"Known validation teams: {len(valid_known_teams)}")
print(f"Unknown validation teams: {len(valid_unknown_teams)}")
print(f"Unknown validation teams list: {sorted(valid_unknown_teams)}")

print()
print(f"Test teams: {len(test_teams)}")
print(f"Known test teams: {len(test_known_teams)}")
print(f"Unknown test teams: {len(test_unknown_teams)}")
print(f"Unknown test teams list: {sorted(test_unknown_teams)}")

Train teams: 121

Validation teams: 96
Known validation teams: 90
Unknown validation teams: 6
Unknown validation teams list: ['Darmstadt', 'FC Heidenheim', 'Frosinone', 'Las Palmas', 'Le Havre', 'Luton']

Test teams: 96
Known test teams: 88
Unknown test teams: 8
Unknown test teams list: ['Como', 'FC Heidenheim', 'Holstein Kiel', 'Ipswich', 'Las Palmas', 'Le Havre', 'Leganes', 'St. Pauli']


### Features and target

In [9]:
feature_columns = [
    "team",
    "opponent",
    "is_home",
    "league",
    "season",
]

target_column = "goals"

X_train = train_df[feature_columns]
y_train = train_df[target_column]

X_valid = valid_df[feature_columns]
y_valid = valid_df[target_column]

X_test = test_df[feature_columns]
y_test = test_df[target_column]

X_train.head()

,team,opponent,is_home,league,season
0,Bordeaux,Nantes,1,Ligue_1,2020
1,Nantes,Bordeaux,0,Ligue_1,2020
2,Dijon,Angers,1,Ligue_1,2020
3,Angers,Dijon,0,Ligue_1,2020
4,Lille,Rennes,1,Ligue_1,2020


## Baseline model

- `OneHotEncoder` для категориальных признаков:
  - `team`
  - `opponent`
  - `league`
  - `season`

- `is_home` оставляем как числовой признак.

Модель:

- `PoissonRegressor`

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import mean_absolute_error, mean_poisson_deviance, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

### Preprocessing

In [11]:
categorical_features = [
    "team",
    "opponent",
    "league",
    "season",
]

numeric_features = [
    "is_home",
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
            categorical_features,
        ),
        (
            "numeric",
            "passthrough",
            numeric_features,
        ),
    ],
    remainder="drop",
)

## Metrics

- `mean_poisson_deviance`— это метрика, которая показывает, насколько хорошо предсказанные лямбды объясняют реальные значения `goals`

Для одного наблюдения:

$$
D(y, \hat{\lambda}) =
2 \left(
y \log\left(\frac{y}{\hat{\lambda}}\right)
- y
+ \hat{\lambda}
\right)
$$

где:

- $y$ — реальное количество голов;
- $\hat{\lambda}$ — предсказанное ожидаемое количество голов.

Для всех наблюдений считается среднее:

$$
\text{Mean Poisson Deviance}
=
\frac{1}{n}
\sum_{i=1}^{n}
2 \left(
y_i \log\left(\frac{y_i}{\hat{\lambda}_i}\right)
- y_i
+ \hat{\lambda}_i
\right)
$$

- `MAE` — средняя абсолютная ошибка;
- `RMSE` — корень из средней квадратичной ошибки.

In [12]:
def evaluate_predictions(y_true, y_pred, dataset_name: str) -> dict:
    """
    Считает основные метрики для предсказанных лямбд.
    
    y_true: реальные голы
    y_pred: предсказанные lambda
    dataset_name: название выборки для вывода
    """
    
    # Poisson deviance требует строго положительные предсказания
    y_pred = np.clip(y_pred, 1e-9, None)
    
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    poisson_deviance = mean_poisson_deviance(y_true, y_pred)
    
    return {
        "dataset": dataset_name,
        "mae": mae,
        "rmse": rmse,
        "mean_poisson_deviance": poisson_deviance,
        "mean_predicted_lambda": float(np.mean(y_pred)),
        "min_predicted_lambda": float(np.min(y_pred)),
        "max_predicted_lambda": float(np.max(y_pred)),
        "mean_actual_goals": float(np.mean(y_true)),
        "min_actual_goals": int(np.min(y_true)),
        "max_actual_goals": int(np.max(y_true)),
    }

In [13]:
import json

In [14]:
experiment_results = pd.DataFrame()


def register_experiment(
    experiment_name: str,
    predictions: dict,
    params: dict | None = None,
    notes: str | None = None,
) -> pd.DataFrame:
    """
    Добавляет результаты эксперимента в общий dataframe.

    experiment_name: уникальное имя эксперимента
    predictions: словарь с предсказаниями для train/validation/test
    params: параметры эксперимента
    notes: короткое описание
    """

    global experiment_results

    params = params or {}

    split_data = {
        "train": (y_train, predictions["train"]),
        "validation": (y_valid, predictions["validation"]),
        "test": (y_test, predictions["test"]),
    }

    rows = []

    for dataset_name, (y_true, y_pred) in split_data.items():
        row = evaluate_predictions(
            y_true=y_true,
            y_pred=y_pred,
            dataset_name=dataset_name,
        )

        row["experiment"] = experiment_name
        row["params"] = json.dumps(params, ensure_ascii=False)
        row["notes"] = notes

        rows.append(row)

    experiment_df = pd.DataFrame(rows)

    # Если ячейку перезапустили, не дублируем старые строки этого же эксперимента
    if not experiment_results.empty:
        experiment_results = experiment_results[
            experiment_results["experiment"] != experiment_name
        ].copy()

    experiment_results = pd.concat(
        [experiment_results, experiment_df],
        ignore_index=True,
    )

    return show_experiment_results()

In [15]:
def show_experiment_results(
    datasets: list[str] | None = None,
    sort_by: str = "mean_poisson_deviance",
    detailed: bool = False,
) -> pd.DataFrame:
    """
    Показывает общий dataframe с результатами экспериментов.

    detailed=False:
        компактная таблица для сравнения моделей.

    detailed=True:
        таблица с params и notes.
    """

    if experiment_results.empty:
        return experiment_results

    result = experiment_results.copy()

    if datasets is not None:
        result = result[result["dataset"].isin(datasets)].copy()

    compact_columns = [
        "experiment",
        "dataset",
        "mae",
        "rmse",
        "mean_poisson_deviance",
        "mean_predicted_lambda",
        "min_predicted_lambda",
        "max_predicted_lambda",
        "mean_actual_goals",
    ]

    detailed_columns = compact_columns + [
        "min_actual_goals",
        "max_actual_goals",
        "params",
        "notes",
    ]

    columns_order = detailed_columns if detailed else compact_columns

    result = result[columns_order]

    return result.sort_values(
        ["dataset", sort_by],
        ascending=[True, True],
    ).reset_index(drop=True)

## Global mean baseline
- всегда предсказывает среднее количество голов из train.

In [16]:
global_train_mean = y_train.mean()

global_mean_predictions = {
    "train": np.full(shape=len(y_train), fill_value=global_train_mean),
    "validation": np.full(shape=len(y_valid), fill_value=global_train_mean),
    "test": np.full(shape=len(y_test), fill_value=global_train_mean),
}

register_experiment(
    experiment_name="global_mean_baseline",
    predictions=global_mean_predictions,
    params={
        "strategy": "global_train_mean",
        "mean": round(float(global_train_mean), 6),
    },
    notes="Always predicts train mean goals.",
)

,experiment,dataset,mae,rmse,mean_poisson_deviance,mean_predicted_lambda,min_predicted_lambda,max_predicted_lambda,mean_actual_goals
0,global_mean_baseline,test,0.992272,1.232509,1.217742,1.396221,1.396221,1.396221,1.413527
1,global_mean_baseline,train,1.003603,1.249498,1.252145,1.396221,1.396221,1.396221,1.396221
2,global_mean_baseline,validation,1.016035,1.260098,1.251250,1.396221,1.396221,1.396221,1.442352


## home/away mean baseline
- предсказывает среднее количество голов с разделением по home/away

In [17]:
home_mean = train_df.loc[train_df["is_home"] == 1, "goals"].mean()
away_mean = train_df.loc[train_df["is_home"] == 0, "goals"].mean()

print(f"Home mean goals from train: {home_mean:.3f}")
print(f"Away mean goals from train: {away_mean:.3f}")


def predict_home_away_mean(X: pd.DataFrame) -> np.ndarray:
    return np.where(
        X["is_home"].to_numpy() == 1,
        home_mean,
        away_mean,
    )

Home mean goals from train: 1.527
Away mean goals from train: 1.265


In [18]:
home_away_mean_predictions = {
    "train": predict_home_away_mean(X_train),
    "validation": predict_home_away_mean(X_valid),
    "test": predict_home_away_mean(X_test),
}

register_experiment(
    experiment_name="home_away_mean_baseline",
    predictions=home_away_mean_predictions,
    params={
        "strategy": "home_away_train_mean",
        "home_mean": round(float(home_mean), 6),
        "away_mean": round(float(away_mean), 6),
    },
    notes="Predicts separate train means for home and away teams.",
)

,experiment,dataset,mae,rmse,mean_poisson_deviance,mean_predicted_lambda,min_predicted_lambda,max_predicted_lambda,mean_actual_goals
0,home_away_mean_baseline,test,0.985534,1.230283,1.213977,1.396221,1.265060,1.527382,1.413527
1,global_mean_baseline,test,0.992272,1.232509,1.217742,1.396221,1.396221,1.396221,1.413527
2,home_away_mean_baseline,train,0.993068,1.242595,1.239805,1.396221,1.265060,1.527382,1.396221
3,global_mean_baseline,train,1.003603,1.249498,1.252145,1.396221,1.396221,1.396221,1.396221
4,home_away_mean_baseline,validation,1.003832,1.251327,1.235855,1.396221,1.265060,1.527382,1.442352
5,global_mean_baseline,validation,1.016035,1.260098,1.251250,1.396221,1.396221,1.396221,1.442352


### Baseline Poisson Regression

In [19]:
alpha = 0.1

poisson_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "regressor",
            PoissonRegressor(
                alpha=alpha,
                max_iter=1000,
            ),
        ),
    ]
)

poisson_model.fit(X_train, y_train)

poisson_predictions = {
    "train": poisson_model.predict(X_train),
    "validation": poisson_model.predict(X_valid),
    "test": poisson_model.predict(X_test),
}

register_experiment(
    experiment_name=f"poisson_regression_alpha_{alpha}",
    predictions=poisson_predictions,
    params={
        "alpha": alpha,
        "features": feature_columns,
    },
    notes="Baseline Poisson Regression with one-hot categorical features.",
)

,experiment,dataset,mae,rmse,mean_poisson_deviance,mean_predicted_lambda,min_predicted_lambda,max_predicted_lambda,mean_actual_goals
0,poisson_regression_alpha_0.1,test,0.969950,1.212283,1.182926,1.395274,1.106371,1.821678,1.413527
1,home_away_mean_baseline,test,0.985534,1.230283,1.213977,1.396221,1.265060,1.527382,1.413527
2,global_mean_baseline,test,0.992272,1.232509,1.217742,1.396221,1.396221,1.396221,1.413527
3,poisson_regression_alpha_0.1,train,0.975193,1.219739,1.200518,1.396260,1.097681,1.877241,1.396221
4,home_away_mean_baseline,train,0.993068,1.242595,1.239805,1.396221,1.265060,1.527382,1.396221
5,global_mean_baseline,train,1.003603,1.249498,1.252145,1.396221,1.396221,1.396221,1.396221
6,poisson_regression_alpha_0.1,validation,0.989026,1.234025,1.205612,1.393937,1.106371,1.821678,1.442352
7,home_away_mean_baseline,validation,1.003832,1.251327,1.235855,1.396221,1.265060,1.527382,1.442352
8,global_mean_baseline,validation,1.016035,1.260098,1.251250,1.396221,1.396221,1.396221,1.442352


### Сравнение

In [20]:
show_experiment_results()

,experiment,dataset,mae,rmse,mean_poisson_deviance,mean_predicted_lambda,min_predicted_lambda,max_predicted_lambda,mean_actual_goals
0,poisson_regression_alpha_0.1,test,0.969950,1.212283,1.182926,1.395274,1.106371,1.821678,1.413527
1,home_away_mean_baseline,test,0.985534,1.230283,1.213977,1.396221,1.265060,1.527382,1.413527
2,global_mean_baseline,test,0.992272,1.232509,1.217742,1.396221,1.396221,1.396221,1.413527
3,poisson_regression_alpha_0.1,train,0.975193,1.219739,1.200518,1.396260,1.097681,1.877241,1.396221
4,home_away_mean_baseline,train,0.993068,1.242595,1.239805,1.396221,1.265060,1.527382,1.396221
5,global_mean_baseline,train,1.003603,1.249498,1.252145,1.396221,1.396221,1.396221,1.396221
6,poisson_regression_alpha_0.1,validation,0.989026,1.234025,1.205612,1.393937,1.106371,1.821678,1.442352
7,home_away_mean_baseline,validation,1.003832,1.251327,1.235855,1.396221,1.265060,1.527382,1.442352
8,global_mean_baseline,validation,1.016035,1.260098,1.251250,1.396221,1.396221,1.396221,1.442352


In [21]:
show_experiment_results(datasets=["validation", "test"])

,experiment,dataset,mae,rmse,mean_poisson_deviance,mean_predicted_lambda,min_predicted_lambda,max_predicted_lambda,mean_actual_goals
0,poisson_regression_alpha_0.1,test,0.969950,1.212283,1.182926,1.395274,1.106371,1.821678,1.413527
1,home_away_mean_baseline,test,0.985534,1.230283,1.213977,1.396221,1.265060,1.527382,1.413527
2,global_mean_baseline,test,0.992272,1.232509,1.217742,1.396221,1.396221,1.396221,1.413527
3,poisson_regression_alpha_0.1,validation,0.989026,1.234025,1.205612,1.393937,1.106371,1.821678,1.442352
4,home_away_mean_baseline,validation,1.003832,1.251327,1.235855,1.396221,1.265060,1.527382,1.442352
5,global_mean_baseline,validation,1.016035,1.260098,1.251250,1.396221,1.396221,1.396221,1.442352


In [22]:
show_experiment_results(datasets=["validation"])

,experiment,dataset,mae,rmse,mean_poisson_deviance,mean_predicted_lambda,min_predicted_lambda,max_predicted_lambda,mean_actual_goals
0,poisson_regression_alpha_0.1,validation,0.989026,1.234025,1.205612,1.393937,1.106371,1.821678,1.442352
1,home_away_mean_baseline,validation,1.003832,1.251327,1.235855,1.396221,1.265060,1.527382,1.442352
2,global_mean_baseline,validation,1.016035,1.260098,1.251250,1.396221,1.396221,1.396221,1.442352


## Alpha tuning

In [33]:
alpha_values = [
    0.0,
    0.0001,
    0.0002,
    0.0003,
    0.0004,
    0.0005,
    0.0006,
    0.0007,
    0.0008,
    0.0009,
    0.0010,
    0.0011,
    0.0012,
    0.0013,
    0.0014,
    0.0015,
    0.0016,
    0.0017,
    0.0018,
    0.0019,
    0.002,
    0.03,
    0.1,
    0.3,
    1.0,
    2.0,
    3.0,
    4.0,
    5.0,
    6.0,
    7.0,
    8.0,
    9.0,
    10.0,
]

alpha_tuning_rows = []

for alpha in alpha_values:
    candidate_model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "regressor",
                PoissonRegressor(
                    alpha=alpha,
                    max_iter=1000,
                ),
            ),
        ]
    )

    candidate_model.fit(X_train, y_train)

    train_pred = candidate_model.predict(X_train)
    valid_pred = candidate_model.predict(X_valid)

    train_metrics = evaluate_predictions(
        y_true=y_train,
        y_pred=train_pred,
        dataset_name="train",
    )

    valid_metrics = evaluate_predictions(
        y_true=y_valid,
        y_pred=valid_pred,
        dataset_name="validation",
    )

    alpha_tuning_rows.append(
        {
            "alpha": alpha,

            "train_mae": train_metrics["mae"],
            "train_rmse": train_metrics["rmse"],
            "train_mean_poisson_deviance": train_metrics["mean_poisson_deviance"],
            "train_min_lambda": train_metrics["min_predicted_lambda"],
            "train_max_lambda": train_metrics["max_predicted_lambda"],

            "valid_mae": valid_metrics["mae"],
            "valid_rmse": valid_metrics["rmse"],
            "valid_mean_poisson_deviance": valid_metrics["mean_poisson_deviance"],
            "valid_min_lambda": valid_metrics["min_predicted_lambda"],
            "valid_max_lambda": valid_metrics["max_predicted_lambda"],
        }
    )

alpha_tuning_results = pd.DataFrame(alpha_tuning_rows)

alpha_tuning_results.sort_values("valid_mean_poisson_deviance").reset_index(drop=True)

,alpha,train_mae,train_rmse,train_mean_poisson_deviance,train_min_lambda,train_max_lambda,valid_mae,valid_rmse,valid_mean_poisson_deviance,valid_min_lambda,valid_max_lambda
0,0.0009,0.890839,1.138043,1.072118,0.397005,4.361248,0.922750,1.176173,1.116745,0.404865,3.499020
1,0.0014,0.891520,1.138846,1.073489,0.439057,4.260335,0.922731,1.176521,1.116778,0.446531,3.447575
2,0.0010,0.890964,1.138180,1.072352,0.404289,4.335871,0.922668,1.176273,1.116791,0.411759,3.483358
3,0.0007,0.890640,1.137787,1.071670,0.379625,4.420394,0.923028,1.176061,1.116815,0.387926,3.532971
4,0.0012,0.891232,1.138498,1.072897,0.420585,4.298859,0.922596,1.176446,1.116842,0.427683,3.467185
5,0.0008,0.890708,1.137868,1.071813,0.384511,4.399156,0.922796,1.176141,1.116849,0.392312,3.517083
6,0.0011,0.891104,1.138343,1.072629,0.412774,4.307143,0.922604,1.176387,1.116853,0.419980,3.469949
7,0.0006,0.890533,1.137645,1.071432,0.368406,4.468064,0.923162,1.176040,1.116926,0.376825,3.560305
8,0.0013,0.891389,1.138680,1.073195,0.428619,4.253591,0.922523,1.176576,1.116939,0.435395,3.443127
9,0.0016,0.891875,1.139248,1.074147,0.454106,4.198694,0.922742,1.176761,1.116967,0.461156,3.417827


In [29]:
best_alpha_row = alpha_tuning_results.sort_values(
    "valid_mean_poisson_deviance"
).iloc[0]

best_alpha = float(best_alpha_row["alpha"])

print(f"Best alpha: {best_alpha}")
print(f"Best validation Poisson deviance: {best_alpha_row['valid_mean_poisson_deviance']:.6f}")

Best alpha: 0.0009
Best validation Poisson deviance: 1.116745


In [30]:
best_poisson_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "regressor",
            PoissonRegressor(
                alpha=best_alpha,
                max_iter=1000,
            ),
        ),
    ]
)

best_poisson_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different

In [31]:
best_poisson_predictions = {
    "train": best_poisson_model.predict(X_train),
    "validation": best_poisson_model.predict(X_valid),
    "test": best_poisson_model.predict(X_test),
}

register_experiment(
    experiment_name=f"poisson_regression_best_alpha",
    predictions=best_poisson_predictions,
    params={
        "alpha": best_alpha,
        "features": feature_columns,
        "selection_metric": "validation_mean_poisson_deviance",
    },
    notes="Best baseline Poisson Regression selected by validation Poisson deviance.",
)

,experiment,dataset,mae,rmse,mean_poisson_deviance,mean_predicted_lambda,min_predicted_lambda,max_predicted_lambda,mean_actual_goals
0,poisson_regression_best_alpha,test,0.918071,1.168509,1.113743,1.413193,0.549101,3.499020,1.413527
1,poisson_regression_alpha_0.1,test,0.969950,1.212283,1.182926,1.395274,1.106371,1.821678,1.413527
2,home_away_mean_baseline,test,0.985534,1.230283,1.213977,1.396221,1.265060,1.527382,1.413527
3,global_mean_baseline,test,0.992272,1.232509,1.217742,1.396221,1.396221,1.396221,1.413527
4,poisson_regression_best_alpha,train,0.890839,1.138043,1.072118,1.396199,0.397005,4.361248,1.396221
5,poisson_regression_alpha_0.1,train,0.975193,1.219739,1.200518,1.396260,1.097681,1.877241,1.396221
6,home_away_mean_baseline,train,0.993068,1.242595,1.239805,1.396221,1.265060,1.527382,1.396221
7,global_mean_baseline,train,1.003603,1.249498,1.252145,1.396221,1.396221,1.396221,1.396221
8,poisson_regression_best_alpha,validation,0.922750,1.176173,1.116745,1.401387,0.404865,3.499020,1.442352
9,poisson_regression_alpha_0.1,validation,0.989026,1.234025,1.205612,1.393937,1.106371,1.821678,1.442352


In [34]:
def get_metric(experiment: str, dataset: str, metric: str = "mean_poisson_deviance") -> float:
    return experiment_results.loc[
        (experiment_results["experiment"] == experiment)
        & (experiment_results["dataset"] == dataset),
        metric,
    ].iloc[0]


for dataset in ["validation", "test"]:
    best_score = get_metric("poisson_regression_best_alpha", dataset)
    global_baseline_score = get_metric("global_mean_baseline", dataset)
    home_away_baseline_score = get_metric("home_away_mean_baseline", dataset)

    global_improvement = (global_baseline_score - best_score) / global_baseline_score * 100
    home_away_improvement = (home_away_baseline_score - best_score) / home_away_baseline_score * 100

    print(f"{dataset}:")
    print(f"  improvement over global mean baseline: {global_improvement:.2f}%")
    print(f"  improvement over home/away mean baseline: {home_away_improvement:.2f}%")
    print()

validation:
  improvement over global mean baseline: 10.75%
  improvement over home/away mean baseline: 9.64%

test:
  improvement over global mean baseline: 8.54%
  improvement over home/away mean baseline: 8.26%



### Сохранение результатов

In [35]:
from pathlib import Path
import joblib

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)

joblib.dump(
    best_poisson_model,
    ARTIFACTS_DIR / "poisson_baseline_model.joblib",
)

experiment_results.to_csv(
    ARTIFACTS_DIR / "poisson_baseline_results.csv",
    index=False,
)

In [36]:
import json

baseline_metadata = {
    "model_name": "poisson_regression_best_alpha",
    "alpha": best_alpha,
    "features": feature_columns,
    "train_seasons": train_seasons,
    "valid_seasons": valid_seasons,
    "test_seasons": test_seasons,
    "selection_metric": "validation_mean_poisson_deviance",
}

with open(ARTIFACTS_DIR / "poisson_baseline_metadata.json", "w", encoding="utf-8") as file:
    json.dump(baseline_metadata, file, ensure_ascii=False, indent=2)